# MHM and MsHHO field equivalence

This notebook compares the finite conforming Galerkin realization of moment-based MsHHO and primal MHM with a polynomial source. Exact-local-space equivalence is a literature result; the local $P_3$ realization here is a finite Galerkin analogue. General reconstructed sources and the face-only $m=-1$ formulation have separate mathematical conventions.


In [ ]:
import numpy as np
from pymhm import TriangleMesh, FaceSpace, SkeletonSpace, solve_darcy, solve_mshho
mesh = TriangleMesh.unit_square()
skeleton = SkeletonSpace(mesh, tuple(FaceSpace.uniform(1) for _ in mesh.faces))
options = dict(skeleton=skeleton, degree=3, local_refinement=2, source=2.0)
mhm = solve_darcy(mesh, **options)
hho = solve_mshho(mesh, cell_degree=1, **options)
error = max(np.max(np.abs(a-b)) for a,b in zip(mhm.pressure,hho.pressure,strict=True))
assert error < 1e-11
error


The complete five-level analytical and four-contrast campaign is acquired separately. The following cells display its current numerical record and figures.
The anisotropic comparison uses explicit wider storage and two original-equation corrections, with the original residual criterion $10^{-10}$. The independently assembled reference uses the same full eight-macrocell physical case. Physical field agreement and insertion into the other rounded operator are distinct checks; at contrast $10^6$ the latter remains outside its declared criterion.
The compact native-verification record identifies the executed Basix modules, source revision and actual bases. The comparison independently assembles its operator and shares the checked SciPy arithmetic.


In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display
root = Path.cwd()
if not (root / "examples").exists():
    root = root.parent
record = json.loads((root / "examples/results/mshho.json").read_text())
assert record["status"] == "verified-gallery"
assert record["mhm_hybrid_refinement_min_steps"] == 2
for row in record["equivalence"]:
    assert row["status"] == "verified"
    assert max(row["original_saddle_diagnostics"][method + "_original_relative_residual"] for method in ("mhm", "mshho")) <= 1e-10
native = json.loads((root / "examples/results/mshho/native-verification.json").read_text())
assert native["criteria"]["changed"] is False
assert native["cross_operator_limit"]["cross_insertion_passed"] is False
assert native["replay"]["all_sampled_fields_and_original_rows_bitwise_equal"] is True
dict(scope=record["scope"], convergence=record["convergence"], equivalence=record["equivalence"], native_reference=native["reference_project"], independent_cross_operator_limit=native["cross_operator_limit"])


In [ ]:
display(Image(filename=str(root / "docs/figures/mshho/convergence.png")))
display(Image(filename=str(root / "docs/figures/mshho/fields.png")))
